[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-empresarial/03-ia-customer-success.ipynb)

# IA en Customer Success: Retención y Predicción de Churn

Herramientas para detectar clientes en riesgo, generar Business Reviews y gestionar tickets con IA.

In [ ]:
import anthropic
import json
from datetime import datetime

client = anthropic.Anthropic()
print("Sistema CS con IA inicializado")

## 1. Dataset de clientes simulado

In [ ]:
CLIENTES = [
    {
        "nombre": "Acme Corp", "plan": "Professional", "mrr": 1200,
        "logins_mes": 8, "logins_mes_anterior": 45,
        "adopcion_pct": 25, "dias_inactivo": 12,
        "tickets_abiertos": 2, "nps": 5,
        "renovacion_dias": 45, "engagement_score": 20
    },
    {
        "nombre": "TechStartup SL", "plan": "Growth", "mrr": 450,
        "logins_mes": 120, "logins_mes_anterior": 110,
        "adopcion_pct": 78, "dias_inactivo": 0,
        "tickets_abiertos": 0, "nps": 9,
        "renovacion_dias": 180, "engagement_score": 90
    },
    {
        "nombre": "Retail Ibérico SA", "plan": "Enterprise", "mrr": 3500,
        "logins_mes": 35, "logins_mes_anterior": 40,
        "adopcion_pct": 55, "dias_inactivo": 3,
        "tickets_abiertos": 1, "nps": 7,
        "renovacion_dias": 30, "engagement_score": 60
    },
    {
        "nombre": "Consultoría XYZ", "plan": "Starter", "mrr": 120,
        "logins_mes": 2, "logins_mes_anterior": 5,
        "adopcion_pct": 15, "dias_inactivo": 20,
        "tickets_abiertos": 0, "nps": None,
        "renovacion_dias": 15, "engagement_score": 10
    },
    {
        "nombre": "FinTech Pro", "plan": "Professional", "mrr": 950,
        "logins_mes": 80, "logins_mes_anterior": 75,
        "adopcion_pct": 65, "dias_inactivo": 1,
        "tickets_abiertos": 0, "nps": 8,
        "renovacion_dias": 90, "engagement_score": 75
    }
]

print(f"Cartera cargada: {len(CLIENTES)} clientes")
mrr_total = sum(c["mrr"] for c in CLIENTES)
print(f"MRR total: {mrr_total:,}€")

## 2. Health Score y segmentación de cartera

In [ ]:
def calcular_health_score(cliente: dict) -> int:
    """Calcula el health score de un cliente (0-100)."""
    pesos = {
        "uso": 0.35,
        "adopcion": 0.25,
        "satisfaccion": 0.20,
        "soporte": 0.10,
        "engagement": 0.10
    }

    nps = cliente.get("nps")
    satisfaccion_score = ((nps + 10) / 20 * 100) if nps is not None else 50

    scores = {
        "uso": min(100, cliente.get("logins_mes", 0) * 2),
        "adopcion": cliente.get("adopcion_pct", 0),
        "satisfaccion": satisfaccion_score,
        "soporte": max(0, 100 - cliente.get("tickets_abiertos", 0) * 20),
        "engagement": cliente.get("engagement_score", 50)
    }

    return round(sum(scores[k] * v for k, v in pesos.items()))


def segmentar_cartera(clientes: list) -> dict:
    """Segmenta la cartera por health score."""
    segmentos = {"champion": [], "saludable": [], "en_riesgo": [], "critico": []}
    for c in clientes:
        score = calcular_health_score(c)
        c = {**c, "health_score": score}
        if score >= 80:
            segmentos["champion"].append(c)
        elif score >= 60:
            segmentos["saludable"].append(c)
        elif score >= 40:
            segmentos["en_riesgo"].append(c)
        else:
            segmentos["critico"].append(c)
    return segmentos


segmentos = segmentar_cartera(CLIENTES)

print("SALUD DE LA CARTERA")
print("=" * 50)
iconos = {"champion": "🏆", "saludable": "✅", "en_riesgo": "⚠️", "critico": "🔴"}

for segmento, clientes_seg in segmentos.items():
    if clientes_seg:
        mrr_seg = sum(c["mrr"] for c in clientes_seg)
        print(f"\n{iconos[segmento]} {segmento.upper()} ({len(clientes_seg)} clientes | {mrr_seg}€ MRR)")
        for c in sorted(clientes_seg, key=lambda x: x["mrr"], reverse=True):
            score = calcular_health_score(c)
            print(f"   • {c['nombre']}: health={score} | MRR={c['mrr']}€ | renovación en {c['renovacion_dias']}d")

## 3. Análisis de riesgo de churn con IA

In [ ]:
def analizar_riesgo_churn(cliente: dict) -> dict:
    """Analiza el riesgo de churn de un cliente con IA."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        system="Eres un especialista en Customer Success con experiencia en predicción de churn SaaS.",
        messages=[{
            "role": "user",
            "content": f"""Evalúa el riesgo de churn de este cliente:

{json.dumps(cliente, ensure_ascii=False, indent=2)}

Devuelve JSON:
{{
  "score_riesgo": 0-100,
  "nivel_riesgo": "crítico/alto/medio/bajo",
  "señales_preocupantes": ["señal 1"],
  "señales_positivas": ["señal 1"],
  "accion_recomendada": "siguiente acción concreta",
  "urgencia": "inmediata/esta_semana/este_mes",
  "argumentos_retencion": ["argumento 1"]
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"analisis": texto}


# Analizar los clientes críticos y en riesgo
clientes_prioritarios = segmentos["critico"] + segmentos["en_riesgo"]

print("ANÁLISIS DE CHURN — CLIENTES PRIORITARIOS")
print("=" * 60)

for cliente in clientes_prioritarios:
    riesgo = analizar_riesgo_churn(cliente)
    print(f"\n🔴 {cliente['nombre']} (MRR: {cliente['mrr']}€)")
    print(f"   Score riesgo: {riesgo.get('score_riesgo', 'N/A')}/100 ({riesgo.get('nivel_riesgo', 'N/A')})")
    print(f"   Urgencia: {riesgo.get('urgencia', 'N/A')}")
    print(f"   Acción: {riesgo.get('accion_recomendada', 'N/A')[:150]}")
    señales_neg = riesgo.get('señales_preocupantes', [])
    if señales_neg:
        print(f"   ⚠️ Señales: {'; '.join(señales_neg[:2])}")

## 4. Generador de QBR (Quarterly Business Review)

In [ ]:
def generar_qbr(cliente: dict, logros: list, objetivos_cliente: list) -> str:
    """Genera el guion de una QBR personalizada."""
    health = calcular_health_score(cliente)

    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1200,
        messages=[{
            "role": "user",
            "content": f"""Genera el guion de una Quarterly Business Review (QBR) para este cliente:

CLIENTE: {cliente['nombre']} | Plan: {cliente['plan']} | MRR: {cliente['mrr']}€
Health Score: {health}/100
Adopción de features: {cliente['adopcion_pct']}%
NPS: {cliente.get('nps', 'no disponible')}

LOGROS DEL TRIMESTRE:
{chr(10).join(f'- {l}' for l in logros)}

OBJETIVOS DECLARADOS DEL CLIENTE:
{chr(10).join(f'- {o}' for o in objetivos_cliente)}

El QBR debe incluir:
1. Apertura personalizada (conecta con sus objetivos)
2. Resumen de uso y adopción
3. 2-3 casos de ROI concreto
4. Plan de mejora si hay bajo uso de features
5. Siguiente paso antes de cerrar la reunión

Tono: consultivo y orientado al negocio del cliente. Máximo 500 palabras."""
        }]
    )
    return resp.content[0].text


# QBR para el cliente enterprise
retail = next(c for c in CLIENTES if c["nombre"] == "Retail Ibérico SA")

qbr = generar_qbr(
    retail,
    logros=[
        "Reducción del 35% en tiempo de gestión de pedidos",
        "Integración exitosa con ERP en Q3",
        "Onboarding completado de 12 usuarios del equipo de operaciones"
    ],
    objetivos_cliente=[
        "Reducir costes operativos un 20% este año",
        "Mejorar la visibilidad del stock en tiempo real",
        "Preparar la expansión a Portugal en Q1 del próximo año"
    ]
)

print("QBR — Retail Ibérico SA")
print("=" * 60)
print(qbr)

## 5. Triaje automático de tickets

In [ ]:
TICKETS_EJEMPLO = [
    {
        "asunto": "¡Urgente! No podemos acceder a la plataforma",
        "mensaje": "Desde hace 2 horas ninguno de nuestros usuarios puede hacer login. Da error 500. Tenemos una demo con un cliente importante en 3 horas."
    },
    {
        "asunto": "¿Cómo exporto los informes a Excel?",
        "mensaje": "Hola, soy nuevo en el equipo y necesito exportar los informes mensuales a Excel para enviárselos a mi jefa. ¿Hay alguna forma de hacerlo?"
    },
    {
        "asunto": "Nos cobráis de más en la factura de noviembre",
        "mensaje": "Hemos revisado la factura 2024-1145 y el importe es de 1.200€ cuando deberíamos pagar 950€ según nuestro contrato. Por favor, corregid."
    }
]

BASE_CONOCIMIENTO = """Exportar a Excel: Dashboard > Reportes > Botón 'Exportar' > Seleccionar Excel.
Login con error 500: Suele ser problema de servidor, escalar a soporte técnico.
Facturación: Verificar en el panel de administración la suscripción activa."""

def clasificar_ticket(ticket: dict) -> dict:
    """Clasifica un ticket y genera respuesta borrador."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        system="Eres un agente de Customer Success experto. Clasificas tickets y generas respuestas empáticas.",
        messages=[{
            "role": "user",
            "content": f"""Procesa este ticket:

ASUNTO: {ticket['asunto']}
MENSAJE: {ticket['mensaje']}
BASE DE CONOCIMIENTO: {BASE_CONOCIMIENTO}

Responde en JSON:
{{
  "categoria": "bug_critico/bug_menor/pregunta_funcionalidad/facturacion/solicitud_feature",
  "prioridad": "P1/P2/P3/P4",
  "sentimiento": "frustrado/neutro/positivo",
  "escalar_a_humano": true,
  "motivo_escalado": "motivo si aplica o null",
  "respuesta_borrador": "texto de respuesta al cliente"
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"respuesta_borrador": texto}


print("TRIAJE AUTOMÁTICO DE TICKETS")
print("=" * 60)

for ticket in TICKETS_EJEMPLO:
    resultado = clasificar_ticket(ticket)
    prioridad_iconos = {"P1": "🔴", "P2": "🟠", "P3": "🟡", "P4": "🟢"}
    icono = prioridad_iconos.get(resultado.get("prioridad", "P3"), "⚪")

    print(f"\n{icono} [{resultado.get('prioridad', 'N/A')}] {ticket['asunto']}")
    print(f"   Categoría: {resultado.get('categoria', 'N/A')} | Sentimiento: {resultado.get('sentimiento', 'N/A')}")
    if resultado.get("escalar_a_humano"):
        print(f"   🚨 ESCALAR: {resultado.get('motivo_escalado', 'requiere atención humana')}")
    print(f"   Respuesta borrador: {resultado.get('respuesta_borrador', '')[:200]}...")